**Training**

In [ ]:
!rm -rf /kaggle/working/SwinIR

In [ ]:
#!git clone https://github.com/kavindamihiran/SwinIR
#!git clone -b kavinda1 https://github.com/kavindamihiran/SwinIR.git
!git clone -b swinllie https://github.com/kavindamihiran/SwinIR.git

In [ ]:
%cd /kaggle/working/SwinIR

In [ ]:
!pip install -r requirements.txt

In [ ]:
# import yaml
# import os

# config_file_path = '/content/SwinIR/configs/swinllie_lol.yaml'

# # Check if the file exists before attempting to read
# if not os.path.exists(config_file_path):
#     print(f"Error: Configuration file not found at {config_file_path}")
# else:
#     with open(config_file_path, 'r') as file:
#         config = yaml.safe_load(file)

#     # Modify the config settings
#     config['training']['epochs'] = 150
#     config['resume']['enabled'] = True
#     config['resume']['checkpoint_path'] = '/content/SwinIR/experiments/test_run/checkpoints/final.pth'

#     # Write the modified config back to the file
#     with open(config_file_path, 'w') as file:
#         yaml.safe_dump(config, file, default_flow_style=False)

#     print(f"Successfully updated {config_file_path}")

In [ ]:
!nvidia-smi

In [ ]:
!python train.py --config configs/swinllie_lol.yaml

In [ ]:
 !rm -rf /kaggle/working/best.pth

In [ ]:
!cp /kaggle/working/SwinIR/experiments/test_run/checkpoints/best.pth /kaggle/working/


## Step 1: Efficiency Metrics (GFLOPs and Inference Time)
Calculates GFLOPs and FPS for a standard resolution (400x600) to prove the model's efficiency compared to heavier architectures.

In [ ]:
!pip install thop
import torch
import time
from thop import profile
import sys
sys.path.append('/kaggle/working/SwinIR')
from swinllie import SwinLLIE

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = SwinLLIE(img_size=128, embed_dim=60, depths=[4, 4, 4], num_heads=[6, 6, 6], window_size=8).to(device)
model.eval()

# Dummy input at 400x600 resolution
dummy_input = torch.randn(1, 3, 400, 600).to(device)

# GFLOPs calculation
flops, params = profile(model, inputs=(dummy_input, ), verbose=False)
print(f"Params: {params / 1e6:.2f} M")
print(f"GFLOPs: {flops / 1e9:.2f} G")

# Inference Time calculation (FPS)
with torch.no_grad():
    # Warmup
    for _ in range(10):
        _ = model(dummy_input)
        
    torch.cuda.synchronize()
    start_time = time.time()
    num_iterations = 50
    for _ in range(num_iterations):
        _ = model(dummy_input)
    torch.cuda.synchronize()
    end_time = time.time()

avg_time = (end_time - start_time) / num_iterations
print(f"Inference Time: {avg_time * 1000:.2f} ms")
print(f"FPS: {1.0 / avg_time:.2f}")

## Step 2: Ablation Study
Trains the model with different loss configurations to prove the effectiveness of each component, especially the Exposure Control Loss.

In [ ]:
import yaml
import os

config_file_path = '/kaggle/working/SwinIR/configs/swinllie_lol.yaml'

def run_ablation(name, lambda_l1=0.0, lambda_vgg=0.0, lambda_color=0.0, lambda_exposure=0.0):
    print(f"\n--- Running Ablation: {name} ---")
    with open(config_file_path, 'r') as file:
        config = yaml.safe_load(file)
        
    config['loss']['lambda_l1'] = lambda_l1
    config['loss']['lambda_vgg'] = lambda_vgg
    config['loss']['lambda_color'] = lambda_color
    config['loss']['lambda_exposure'] = lambda_exposure
    
    config['training']['epochs'] = 50 # Reduced for ablation speed
    config['training']['save_dir'] = f"./experiments/ablation_{name}"
    
    with open(config_file_path, 'w') as file:
        yaml.safe_dump(config, file, default_flow_style=False)
        
    # Run training
    os.system(f"python train.py --config {config_file_path}")

# Run the 4 ablation models
# 1. Base (L1 loss only)
run_ablation('base_l1', lambda_l1=1.0)
# 2. Base + VGG + Color
run_ablation('base_vgg_color', lambda_l1=1.0, lambda_vgg=0.1, lambda_color=0.5)
# 3. Base + All Losses (No Exposure Loss)
run_ablation('no_exposure', lambda_l1=1.0, lambda_vgg=0.1, lambda_color=0.5, lambda_exposure=0.0)
# 4. Ours (Full Loss)
run_ablation('full', lambda_l1=1.0, lambda_vgg=0.1, lambda_color=0.5, lambda_exposure=1.0)

## Step 3: Evaluate on LOL-v2 Dataset
Evaluates the trained model on the LOL-v2 dataset to prove generalization across different datasets.
**Note**: Make sure to add the `tanhyml/lol-v2-dataset` dataset to this Kaggle notebook.

In [ ]:
import os

lol_v2_path = '/kaggle/input/lol-v2-dataset/LOL-v2'
if os.path.exists(lol_v2_path):
    print("Evaluating on LOL-v2 dataset...")
    
    # Create a compatible structure for check_metrics.py ('generic' dataset type expects test/low and test/high)
    !mkdir -p ./datasets/LOL-v2/test/low ./datasets/LOL-v2/test/high
    !cp -r /kaggle/input/lol-v2-dataset/LOL-v2/Real_captured/Test/Low/* ./datasets/LOL-v2/test/low/
    !cp -r /kaggle/input/lol-v2-dataset/LOL-v2/Real_captured/Test/High/* ./datasets/LOL-v2/test/high/
    
    # Run inference and check metrics
    !python inference.py --input ./datasets/LOL-v2/test/low --output ./results_lolv2 --weights ./experiments/test_run/checkpoints/best.pth
    
    # Note: For check_metrics.py, we temporarily change dataset_type to 'generic' to avoid LOL's eval15 hardcoding
    !sed -i "s/get_dataloader('lol'/get_dataloader('generic'/g" check_metrics.py
    !python check_metrics.py --dataset ./datasets/LOL-v2 --weights ./experiments/test_run/checkpoints/best.pth
else:
    print("Please click 'Add Data' on the right panel and search for 'tanhyml/lol-v2-dataset' to attach it.")

## Step 5: Implementation Details (For the paper)
- **GPU Hardware**: Display the exact GPU used in Kaggle (usually 1x NVIDIA Tesla T4 or P100).
- **Software Stack**: PyTorch 2.0+ with CUDA.
Be sure to record the training time displayed above to add to your paper!

In [ ]:
!nvidia-smi
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")